In [1]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


In [12]:
from statistics import mean
import pandas as pd

from scripts.main import load_graph_and_communities


datasets = [
    'Algebra',
    'Geometry',
    'Restaurants-Rev',
    'Music-Rev',
    'Bars-Rev',
    'contact-primary-school'
]

rows = []

for dataset in datasets:
    graph, comm = load_graph_and_communities(dataset, tau=None, verbose=False, source=False)

    # Basic counts
    n_nodes = len(graph.nodes)
    n_hyperedges = len(graph.hyperedges)

    # Hyperedge sizes
    hyperedge_sizes = [len(hedge_nodes) for hedge_nodes in graph.hyperedges.values()]
    avg_hyperedge_size = mean(hyperedge_sizes) if hyperedge_sizes else 0
    max_hyperedge_size = max(hyperedge_sizes) if hyperedge_sizes else 0

    # Projected node degree + node hyperdegree
    neighbors = {node: set() for node in graph.nodes}
    node_hyperdegrees = {node: 0 for node in graph.nodes}

    for hedge_nodes in graph.hyperedges.values():
        for node in hedge_nodes:
            neighbors[node].update(hedge_nodes - {node})
            node_hyperdegrees[node] += 1

    degree_values = [len(neighbors[node]) for node in graph.nodes]
    hyperdegree_values = list(node_hyperdegrees.values())

    avg_node_degree = mean(degree_values) if degree_values else 0
    max_node_degree = max(degree_values) if degree_values else 0
    avg_node_hyperdegree = mean(hyperdegree_values) if hyperdegree_values else 0
    max_node_hyperdegree = max(hyperdegree_values) if hyperdegree_values else 0

    rows.append({
        "Dataset": dataset,
        "Nodes": n_nodes,
        "Hyperedges": n_hyperedges,
        "Avg. hyperedge size": round(avg_hyperedge_size, 2),
        "Max. hyperedge size": max_hyperedge_size,
        "Avg. node degree": round(avg_node_degree, 2),
        "Max. node degree": max_node_degree,
    })

summary_df = pd.DataFrame(rows)

# Save for Overleaf / Excel / manual table copy
summary_df.to_csv("dataset_summary.csv", index=False)

summary_df

,Dataset,Nodes,Hyperedges,Avg. hyperedge size,Max. hyperedge size,Avg. node degree,Max. node degree
0,Algebra,423,1268,6.52,107,78.90,303
1,Geometry,580,1193,10.47,230,164.79,474
2,Restaurants-Rev,565,601,7.66,43,79.75,310
3,Music-Rev,1106,694,15.13,83,167.88,865
4,Bars-Rev,1234,1194,9.94,73,174.30,818
5,contact-primary-school,242,12704,2.42,5,68.74,134


In [16]:
from statistics import mean, median, stdev
import pandas as pd

from scripts.main import load_graph_and_communities


datasets = [
    'Algebra',
    'Geometry',
    'Restaurants-Rev',
    'Music-Rev',
    'Bars-Rev',
    'contact-primary-school'
]

taus = [
    (None, "ground_truth"),
    (0, "tau_0"),
    (2, "tau_2"),
]

rows = []

for dataset in datasets:
    for tau, tau_name in taus:
        graph, comm = load_graph_and_communities(
            dataset,
            tau=tau,
            verbose=False,
            source=False
        )

        # community sizes
        sizes = [len(nodes) for nodes in comm.values()]

        if not sizes:
            continue

        n_nodes = len(graph.nodes)
        n_communities = len(sizes)

        avg_size = mean(sizes)
        median_size = median(sizes)
        max_size = max(sizes)
        std_size = stdev(sizes) if len(sizes) > 1 else 0

        singleton_frac = sum(1 for s in sizes if s == 1) / n_communities
        largest_frac = max_size / n_nodes

        rows.append({
            "Dataset": dataset,
            "Tau": tau_name,
            "Communities": n_communities,
            "Avg. size": round(avg_size, 2),
        })

df = pd.DataFrame(rows)

df_pivot = df.pivot(
    index="Dataset",
    columns="Tau",
    values=["Communities", "Avg. size"]
)

# Flatten column names
df_pivot.columns = [
    f"{col1}_{col2}" for col1, col2 in df_pivot.columns
]

df_pivot = df_pivot.reset_index()

# Order columns nicely
df_pivot = df_pivot[[
    "Dataset",
    "Communities_ground_truth", "Avg. size_ground_truth",
    "Communities_tau_0", "Avg. size_tau_0",
    "Communities_tau_2", "Avg. size_tau_2",
]]

# Save
df_pivot.to_csv("community_summary_table.csv", index=False)

df_pivot


,Dataset,Communities_ground_truth,Avg. size_ground_truth,Communities_tau_0,Avg. size_tau_0,Communities_tau_2,Avg. size_tau_2
0,Algebra,32.0,29.53,17.0,24.88,10.0,42.30
1,Bars-Rev,15.0,91.67,71.0,17.38,37.0,33.35
2,Geometry,25.0,48.76,30.0,19.33,21.0,27.62
3,Music-Rev,7.0,171.43,63.0,17.56,24.0,46.08
4,Restaurants-Rev,9.0,77.00,13.0,43.46,7.0,80.71
5,contact-primary-school,11.0,22.00,9.0,26.89,6.0,40.33


In [ ]:
import matplotlib.pyplot as plt

for dataset, degrees in hyperdegree_distributions.items():
    counts = list(degrees.values())
    max_degree = max(counts) if counts else 0
    bins = range(0, max_degree + 2)
    plt.figure(figsize=(8, 4))
    plt.hist(counts, bins=bins, edgecolor='black', align='left')
    plt.yscale('log')
    plt.title(f'Degree distribution for {dataset}')
    plt.xlabel('Number of neighboring nodes')
    plt.ylabel('Node count (log scale)')
    plt.xticks(range(0, max_degree + 1, max(1, max_degree // 10)))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()